# Data Compression Program Documentation

This documentation provides a comprehensive overview of the functions implemented in the data compression program, which efficiently compresses text data using Run-Length Encoding (RLE) and Huffman Coding. The documentation is split into two parts: initial setup in Notebook 1 and data processing in Notebook 2.

## Notebook 1: Setup and Configuration

### Overview
**Purpose**:
- Captures user input for the dataset path and preferred output method.
- Sets the environment for the data compression tasks that follow in Notebook 2.

**Process**:
1. **Prompt for Dataset Path**:
   - Users can enter a custom path for their dataset or use a default path provided by the system.
2. **Select Output Preference**:
   - Users choose how they want the output handled: printed to console, saved to file, or both.

### User Interaction
- **Dataset Path Input**: Users are prompted to input the path to their dataset file or accept a default path.
- **Output Type Selection**: Users select their output preference from the following options:
  1. Print to Console
  2. Save to File
  3. Both Print and Save

These settings are stored using IPython's `%store` magic command to make them accessible in Notebook 2.

---

## Functions Overview in Notebook 2

### Run-Length Encoding Function

**Purpose**:
- Compresses a string using Run-Length Encoding, which is effective for data with repeated characters.

**Input**:
- `data` (string): The string to be encoded.

**Output**:
- Returns an encoded string where sequences of the same character are replaced by the character followed by the count.

**Process**:
1. Initialize an empty list `encoding`.
2. Check if `data` is empty; if yes, return an empty string.
3. Set `prev_char` to the first character of `data` and `count` to 1.
4. Iterate through `data` starting from the second character:
   - If the current character is the same as `prev_char`, increment `count`.
   - If different, append `prev_char` followed by `count` to `encoding`, update `prev_char` to the current character, and reset `count` to 1.
5. After the loop, append the last `prev_char` and `count` to `encoding`.
6. Convert `encoding` list to a string and return.

### Huffman Tree Creation Function

**Purpose**:
- Builds a Huffman tree to determine optimal prefix-free codes based on character frequencies, essential for efficient data compression.

**Input**:
- `frequency` (dictionary): A dictionary where keys are characters and values are their respective frequencies.

**Output**:
- Returns a list representing the Huffman tree with encoded symbols.

**Process**:
1. Convert `frequency` dictionary into a list of nodes, each node is `[weight, [symbol, ""]]`.
2. Create a min-heap from the list using `heapq.heapify()`.
3. While there are more than one node in the heap:
   - Remove the two nodes with the lowest frequency.
   - For each symbol in these nodes, append '0' to the codes from one node and '1' to the other.
   - Merge these two nodes and push the result back into the heap.
4. Return the root of the tree containing all the Huffman codes.

### Huffman Coding Function

**Purpose**:
- Generates Huffman codes for characters based on their frequencies in the data, optimizing the encoded data length.

**Input**:
- `data` (string): The string for which Huffman codes need to be generated.

**Output**:
- Returns a dictionary of Huffman codes where keys are characters and values are their codes.

**Process**:
1. Use `Counter` from `collections` to calculate the frequency of each character in `data`.
2. Call `create_tree` function with these frequencies to get the Huffman tree.
3. Extract Huffman codes from the tree and return them.

### Integrated Compression Function

**Purpose**:
- Compresses data using both Run-Length Encoding and Huffman Coding to maximize data compression efficiency.

**Input**:
- `data` (string): The string to be compressed.

**Output**:
- Returns a tuple containing the dictionary of Huffman codes and the final compressed string.

**Process**:
1. Compress `data` using Run-Length Encoding and store the result.
2. Generate Huffman codes for the RLE output.
3. Encode the RLE output using the generated Huffman codes and concatenate the results.
4. Return the Huffman codes and the encoded string.


In [ ]:
# Initial code
'''
import heapq
import os
import sys
from collections import Counter

# Run-Length Encoding Function
def run_length_encode(data):
    """
    Encodes the input string using Run-Length Encoding (RLE).
    
    Parameters:
    data (str): The input string to be encoded.

    Returns:
    str: The RLE encoded string.
    """
    if not data:
        return ""  # Return an empty string if the input data is empty
    encoding = []  # List to hold the RLE pairs
    prev_char = data[0]  # Initialize with the first character of the string
    count = 1  # Initialize the count of the first character

    # Iterate over the input string starting from the second character
    for char in data[1:]:
        if char == prev_char:
            count += 1  # Increment the count if the current char is the same as previous
        else:
            # If the current char is different, append the previous char and its count to the list
            encoding.append(f"{prev_char}{count}")
            prev_char = char  # Update the prev_char to current char
            count = 1  # Reset the count for the new character
    encoding.append(f"{prev_char}{count}")  # Append the last character and its count
    return ''.join(encoding)  # Join the list into a single string and return

# Function to create Huffman tree
def create_tree(frequency):
    """
    Creates a Huffman tree for given frequencies using a min-heap.

    Parameters:
    frequency (dict): A dictionary of frequencies where keys are characters and values are their frequencies.

    Returns:
    list: A list containing the Huffman tree with encoded symbols.
    """
    heap = [[weight, [symbol, ""]] for symbol, weight in frequency.items()]
    heapq.heapify(heap)  # Transform the list of frequencies into a heap

    # Iterate while there is more than one node in the heap
    while len(heap) > 1:
        lo = heapq.heappop(heap)  # Pop the two nodes with the lowest frequency
        hi = heapq.heappop(heap)

        # Add '0' to the codes of the lower frequency node and '1' to the codes of the higher frequency node
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]

        # Push the new combined node back into the heap
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    return heap[0][1:]  # Return the root node containing all the Huffman codes

# Function to generate Huffman codes
def huffman_code(data):
    """
    Generates Huffman codes for encoding a string.

    Parameters:
    data (str): The input string for which Huffman codes are to be generated.

    Returns:
    dict: A dictionary where keys are the characters and values are their respective Huffman codes.
    """
    frequency = Counter(data)  # Count the frequency of each character in the string
    tree = create_tree(frequency)  # Create a Huffman tree from these frequencies
    huffman_code = {symbol: code for symbol, code in tree}  # Extract the codes from the tree
    return huffman_code

# Integrated Compression Function
def compress_data(data):
    """
    Compresses the input data first using Run-Length Encoding and then Huffman Coding.

    Parameters:
    data (str): The input string to be compressed.

    Returns:
    tuple: A tuple containing the Huffman codes and the compressed data.
    """
    rle_encoded = run_length_encode(data)  # Perform Run-Length Encoding
    huffman_coded = huffman_code(rle_encoded)  # Generate Huffman codes for the RLE output
    huffman_encoded_output = ''.join(huffman_coded[char] for char in rle_encoded)  # Encode the RLE output using Huffman codes
    return huffman_coded, huffman_encoded_output

# Read data from a file
def read_data_from_file(file_path):
    """
    Reads data from a file.

    Parameters:
    file_path (str): The path to the file to be read.

    Returns:
    str: The content of the file as a string.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        data = file.read()  # Read all data from the file
    return data

# Write output to a file
def write_output_to_file(file_path, output):
    """
    Writes data to a file.

    Parameters:
    file_path (str): The path to the file where data will be written.
    output (str): The data to write.
    """
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(output)  # Write the output data to the file

# Main function to handle the pipeline with command-line arguments
def main():
    """
    Main function to control the flow of data compression from command-line inputs.
    """
    if len(sys.argv) < 3:
        print("Usage: python script.py input_file_path output_file_path [print_output]")
        sys.exit(1)  # Exit if not enough command line arguments are provided
    
    input_file_path = sys.argv[1]  # Input file path
    output_file_path = sys.argv[2]  # Output file path
    print_output = sys.argv[3].lower() == 'true' if len(sys.argv) > 3 else False  # Optional print output

    try:
        data = read_data_from_file(input_file_path)  # Read data from the specified input file
        _, compressed_data = compress_data(data)  # Compress the data

        write_output_to_file(output_file_path, compressed_data)  # Write the compressed data to the specified output file

        if print_output:
            print("Compressed Data Output:")
            print(compressed_data)  # Optionally print the compressed data

        print(f"Compression complete. Output saved to {output_file_path}")

    except Exception as e:
        print(f"An error occurred: {e}")
        sys.exit(1)  # Exit if an error occurs

if __name__ == '__main__':
    main()
'''

In [ ]:
# Import necessary libraries
import IPython  # Import IPython for using its magic commands like %store.

# Function to get the dataset path from the user
def get_dataset_path():
    """
    Prompts the user to specify a path for the dataset file.

    The function offers a default path but allows the user to enter a custom path. It ensures that any leading
    or trailing whitespace in the user input is removed. If the user presses enter without typing anything,
    the default path is used.

    Returns:
        str: The path to the dataset file, either entered by the user or the predefined default.
    """
    default_path = '/path/to/default/dataset.csv'  # Default dataset path if the user does not provide one.
    # Prompt the user to input a dataset path or use the default by just pressing enter.
    prompt = f"Enter dataset path or hit enter to use default ({default_path}): "
    path = input(prompt).strip()  # Remove any extra whitespace from the input for cleanliness.
    return path if path else default_path  # Return the user-specified path or the default if no input was given.

# Function to get user's preference for output type
def get_output_preference():
    """
    Asks the user to select how they prefer the output to be handled.

    This function presents the user with three output handling options and expects an integer input corresponding
    to the user's choice. It ensures that the input is stripped of any extraneous whitespace and converted to an integer.

    Returns:
        int: The user's output preference as an integer (1, 2, or 3).
    """
    print("Select the output type:")
    print("1: Print to Console")  # Option 1: Output results to the console.
    print("2: Save to File")      # Option 2: Save results to a file.
    print("3: Both Print and Save")  # Option 3: Both print to the console and save to a file.
    # Prompt the user to enter their choice based on the options presented.
    option = input("Enter your choice (1, 2, or 3): ").strip()
    return int(option)  # Convert the input to an integer to simplify further processing.

# Main function to gather inputs and store them for use in another notebook
def setup_environment():
    """
    Sets up the environment by gathering necessary inputs from the user and storing them.

    This function calls other functions to get the dataset path and output preferences, and then stores these
    using IPython's %store magic command for accessibility in subsequent notebooks. It also prints the
    configurations to confirm to the user what settings have been applied.
    """
    dataset_path = get_dataset_path()  # Get the dataset path from the user.
    output_preference = get_output_preference()  # Get the user's output preference.

    # Use IPython's %store command to make the dataset path and output preference available in other notebooks.
    %store dataset_path
    %store output_preference
    
    # Print the set configurations to provide feedback to the user about the stored settings.
    print(f"Configuration set:\nDataset path: {dataset_path}\nOutput preference: {output_preference}")

# Execute the setup environment function when the script is run directly.
if __name__ == "__main__":
    setup_environment()


In [ ]:
""" changelog: added integrated unitest framework, large file and astraming data support and modified the main function to use chunked reading"""
import heapq
import os
import sys
from collections import Counter
import unittest

# Function to perform Run-Length Encoding
def run_length_encode(data):
    """
    Encodes the input string using Run-Length Encoding (RLE).

    This method compresses the data by reducing sequences of the same character 
    to the character followed by the number of occurrences. This is particularly 
    effective for data with many consecutive repeated characters.

    Parameters:
    data (str): The input string to be encoded.

    Returns:
    str: The RLE encoded string.
    """
    if not data:
        return ""  # Handle the edge case where the input data is empty
    encoding = []  # Initialize an empty list to store character counts
    prev_char = data[0]  # Start with the first character
    count = 1  # Counter for occurrences of each character

    # Iterate through the input string starting from the second character
    for char in data[1:]:
        if char == prev_char:
            count += 1  # Increment the count if the current char is the same as previous
        else:
            encoding.append(f"{prev_char}{count}")  # Append the count and character to the list
            prev_char = char  # Update the prev_char to current character
            count = 1  # Reset the count for the new character
    encoding.append(f"{prev_char}{count}")  # Append the last character and its count
    return ''.join(encoding)  # Convert the list to a string and return

# Function to create Huffman tree
def create_tree(frequency):
    """
    Creates a Huffman tree using the frequencies of characters.

    This method uses a min-heap to build the tree efficiently, ensuring that 
    characters with the lowest frequencies are combined first, which is essential 
    for optimal prefix-free encoding.

    Parameters:
    frequency (dict): A dictionary with characters as keys and their frequencies as values.

    Returns:
    list: A list representing the Huffman tree with encoded symbols.
    """
    heap = [[weight, [symbol, ""]] for symbol, weight in frequency.items()]
    heapq.heapify(heap)  # Transform list of frequencies into a heap for efficient access

    # Build the tree by combining the two least frequent elements until one element remains
    while len(heap) > 1:
        lo = heapq.heappop(heap)  # Pop the two nodes with the lowest frequency
        hi = heapq.heappop(heap)

        # Append '0' to the codes of the lower frequency node and '1' to the higher
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]

        # Merge the two nodes and push back to the heap
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    return heap[0][1:]  # Return the root node containing all the Huffman codes

# Function to generate Huffman codes
def huffman_code(data):
    """
    Generates Huffman codes for encoding a string.

    This function calculates the frequency of each character in the input data, builds a Huffman tree, 
    and then extracts the Huffman codes for each character. These codes are optimal for data compression 
    as they ensure the most frequent characters have the shortest codes.

    Parameters:
    data (str): The input string for which Huffman codes need to be generated.

    Returns:
    dict: A dictionary where keys are the characters and values are their respective Huffman codes.
    """
    frequency = Counter(data)  # Calculate the frequency of each character in the string
    tree = create_tree(frequency)  # Build the Huffman tree from these frequencies
    huffman_code = {symbol: code for symbol, code in tree}  # Extract the codes from the tree
    return huffman_code

# Integrated Compression Function
def compress_data(data):
    """
    Compresses the input data first using Run-Length Encoding and then Huffman Coding.

    This function integrates two compression methods to enhance the compression rate. First,
    Run-Length Encoding is applied to reduce the size by compressing sequential repetitions of characters.
    Then, Huffman Coding is applied to further compress the data by using variable-length codes for characters
    based on their frequencies.

    Parameters:
    data (str): The input string to be compressed.

    Returns:
    tuple: A tuple containing the Huffman codes and the compressed data.
    """
    rle_encoded = run_length_encode(data)  # Perform Run-Length Encoding
    huffman_coded = huffman_code(rle_encoded)  # Generate Huffman codes for the RLE output
    huffman_encoded_output = ''.join(huffman_coded[char] for char in rle_encoded)  # Encode the RLE output using Huffman codes
    return huffman_coded, huffman_encoded_output

# Generator to read data from a file in chunks
def read_data_from_file_in_chunks(file_path, chunk_size=1024):
    """
    Reads data from a file in specified chunks. This approach is beneficial for handling
    large files by not loading the entire file into memory at once.

    Parameters:
    file_path (str): The path to the file to be read.
    chunk_size (int): The size of each chunk to read (default 1024).

    Yields:
    str: A chunk of data from the file.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        while True:
            data = file.read(chunk_size)
            if not data:
                break
            yield data

# Writes data to a file using a generator for chunked data input
def write_output_to_file_in_chunks(file_path, data_generator):
    """
    Writes data to a file from a generator, which provides data in chunks. This method is efficient
    for writing large datasets by writing in segments and minimizing memory usage.

    Parameters:
    file_path (str): The path to the file where data will be written.
    data_generator (generator): Generator that yields chunks of data to write.
    """
    with open(file_path, 'w', encoding='utf-8') as file:
        for chunk in data_generator:
            file.write(chunk)

# Main function modified to use chunked reading
def main():
    """
    Main function to control the flow of data compression from command-line inputs.
    Handles command-line arguments, reads data in chunks, compresses it, and writes the output.

    Usage:
    python script.py input_file_path output_file_path [print_output]
    """
    if len(sys.argv) < 3:
        print("Usage: python script.py input_file_path output_file_path [print_output]")
        sys.exit(1)
    
    input_file_path = sys.argv[1]
    output_file_path = sys.argv[2]
    print_output = sys.argv[3].lower() == 'true' if len(sys.argv) > 3 else False

    try:
        data_generator = read_data_from_file_in_chunks(input_file_path)
        compressed_data = ""
        for data in data_generator:
            _, compressed_chunk = compress_data(data)
            compressed_data += compressed_chunk
        write_output_to_file_in_chunks(output_file_path, iter([compressed_data]))

        if print_output:
            print("Compressed Data Output:")
            print(compressed_data)

        print(f"Compression complete. Output saved to {output_file_path}")

    except Exception as e:
        print(f"An error occurred: {e}")
        sys.exit(1)

# Unit tests for the compression functions
class TestCompressionAlgorithms(unittest.TestCase):
    def test_run_length_encode_empty(self):
        self.assertEqual(run_length_encode(""), "")

    def test_run_length_encode_simple(self):
        self.assertEqual(run_length_encode("aaabb"), "a3b2")

    def test_huffman_code_simple(self):
        result = huffman_code("aaabb")
        self.assertTrue(isinstance(result, dict))
        self.assertEqual(len(result), 2)

    def test_compress_data_integration(self):
        codes, compressed = compress_data("aaabb")
        self.assertTrue(isinstance(compressed, str))
        self.assertTrue(isinstance(codes, dict))
        self.assertNotEqual(compressed, "")

if __name__ == '__main__':
    if len(sys.argv) > 1 and sys.argv[1] == 'test':
        unittest.main(argv=['first-arg-is-ignored'], exit=False)
    else:
        main()
